# Collections in depth and comprehensions

Collections stop being just storage containers at this level. They become part of how you express an algorithm. Lists, dictionaries, sets, and comprehensions let you shape data in compact ways, but compact code is only good when the underlying idea stays readable.

Comprehensions are especially important because they combine iteration, filtering, and transformation in one expression. Used well, they make intent obvious. Used badly, they hide logic inside a dense one-liner.

The aim of this module is to make you deliberate about both data shape and performance. When you choose a collection or a comprehension style, you are choosing both readability and computational behaviour.

## Visual model

```text
input data -> filter -> transform -> group -> result
```

## How to use this notebook

Read the concept notes first, then run the code cells one at a time. After each run, change an input, prediction, or line of code and rerun it. Intermediate Python becomes easier when you treat every notebook as a place to test a mental model, not just a place to read finished answers.

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.


---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. `dict`: a hash table with compact ordering

Since 3.6, CPython's dict has two parts: a dense array of entries in insertion
order, and a sparse index array of positions into it.

```text
indices : [ _ , 1 , _ , 0 , _ , 2 , _ , _ ]    sparse, sized to load factor
entries : [ (hash, key, value),                dense, INSERTION ORDER
            (hash, key, value),
            (hash, key, value) ]
```


That layout gives ordering for free and saves memory. Insertion order became a
**language guarantee in 3.7** (it was a CPython implementation detail in 3.6 —
worth knowing when reading old code).

| Operation | Complexity |
|---|---|
| `d[k]`, `d[k] = v`, `del d[k]`, `k in d` | O(1) average, O(n) worst |
| iteration | O(n), insertion order |
| `len` | O(1) |

The lookup: hash the key, mask it to an index, probe. On collision, probe again.
On a match of hashes, confirm with `==`. This is why **`__hash__` and `__eq__`
must agree** (Module 09), and why a mutable key would be unfindable (Module 03).

### The methods worth knowing

```text
d.get(k, default)               # no KeyError
d.setdefault(k, [])             # get, inserting the default if absent
d.pop(k, default)
d.popitem()                     # removes and returns the LAST item (LIFO)
d | other                       # merge, 3.9+ (right wins)
d |= other                      # in-place merge
{**a, **b}                      # merge, older syntax
d.keys() / .values() / .items() # VIEWS: live, not copies
dict.fromkeys(seq)              # dedupe preserving order (Module 03)
```


Views are live and cheap:

In [ ]:
keys = d.keys()
d["new"] = 1
print("new" in keys)      # True -- the view reflects the change

Key views also support set operations: `d1.keys() & d2.keys()` gives the common
keys. `d.items() - other.items()` gives the differing pairs. Underused and
excellent.

**`setdefault` versus `defaultdict`:**

In [ ]:
groups = {}
for item in items:
    groups.setdefault(item.kind, []).append(item)     # fine

from collections import defaultdict
groups = defaultdict(list)
for item in items:
    groups[item.kind].append(item)                     # cleaner

The `defaultdict` catch: **reading a missing key inserts it.** `if x in dd`
is safe; `dd[x]` is not. Convert with `dict(dd)` before returning it to code
that does not expect that behaviour.

---

## Concept 5. Comprehensions

In [ ]:
[f(x) for x in xs if pred(x)]           # list
{f(x) for x in xs}                      # set
{k: v for k, v in pairs}                # dict
(f(x) for x in xs)                      # GENERATOR -- lazy, not a tuple

Read them outside-in: *what to produce*, then *what to loop over*, then *what to
keep*.

In [ ]:
[y for x in matrix for y in x]           # flatten: loops in the same order
                                          # you would write them nested
[[y for y in row] for row in matrix]     # nested comprehension: inner produces
                                          # a list per row

The multi-`for` order trips everyone up once. It reads left to right in the same
order as the equivalent nested `for` statements.

### Conditions

In [ ]:
[x for x in xs if x > 0]                 # FILTER: after the for
[x if x > 0 else 0 for x in xs]          # TRANSFORM: a conditional expression
                                          # before the for
[x for x in xs if x > 0 if x < 10]       # two filters, ANDed

### When not to use one

- More than two `for` clauses, or a `for` plus two conditions: use a loop.
- Any side effect. `[print(x) for x in xs]` builds a list of `None` and throws
  it away. Write a `for` loop.
- When the expression no longer fits on a line and reads worse than three lines
  of loop.

A comprehension should read as a *description of the result*. When it starts
reading as a *procedure*, it should be a loop.

### Generator expressions: the lazy version

In [ ]:
sum(x**2 for x in range(1_000_000))     # never builds the list
any(line.startswith("ERROR") for line in fh)   # stops at the first hit
max((score(x), x) for x in candidates)

Parentheses are optional when it is the only argument to a call. Use a generator
expression when you are consuming the values once — it uses O(1) memory instead
of O(n) and can short-circuit. Module 14 makes this a design tool.

---

## Concept 6. Sorting

In [ ]:
sorted(xs)                                   # new list
xs.sort()                                    # in place, returns None
sorted(xs, key=len)                          # by a computed value
sorted(xs, key=lambda p: (p.dept, -p.score)) # multi-key; - reverses a number
sorted(xs, reverse=True)
sorted(xs, key=str.casefold)                 # case-insensitive text

from operator import attrgetter, itemgetter
sorted(people, key=attrgetter("age"))        # faster and clearer than a lambda
sorted(rows, key=itemgetter(1, 0))

Facts to keep:

- **Timsort, O(n log n), and stable.** Stability means equal elements keep their
  relative order, which is what makes multi-pass sorting work:

  ```python
  rows.sort(key=itemgetter("name"))     # secondary key first
  rows.sort(key=itemgetter("dept"))     # primary key last
  ```

- The `key` function is called **once per element**, not on every comparison.
  That is why `key=` beats the removed `cmp=` and why an expensive key is fine.
- For reverse-sorting on a non-numeric key, use `reverse=True` rather than
  negating — you cannot negate a string.
- For top-k, `heapq.nlargest(k, xs)` is O(n log k) and streams its input.

---

## Concept 8. Choosing a container

| Need | Use |
|---|---|
| Ordered, changes | `list` |
| Fixed record, hashable | `tuple` / `NamedTuple` / frozen dataclass |
| Lookup by key | `dict` |
| Membership, dedupe, set algebra | `set` |
| Queue, sliding window, last-N | `deque` |
| Counting | `Counter` |
| Grouping | `defaultdict(list)` |
| Priority / top-k | `heapq` |
| Layered config | `ChainMap` |
| Sorted, with fast inserts | `bisect` on a list, or `sortedcontainers` |
| Large numeric data | `array`, or NumPy (Module 29) |

The two questions that answer this almost every time:

1. **How will I look things up?** By position → list. By key → dict. By presence
   → set.
2. **Where do I add and remove?** Both ends → deque. End only → list.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: `list`: a dynamic array of pointers
- Section 2: `dict`: a hash table with compact ordering
- Section 3: `set`: a hash table without values
- Section 4: `tuple`
- Section 5: Comprehensions
- Section 6: Sorting
- Section 7: `collections`
- Section 8: Choosing a container

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from collections.abc import Iterable, Iterator
from typing import Any

WORDS = ["apple", "Banana", "cherry", "date", "Elderberry", "fig"]
NUMS = [3, -1, 4, -1, 5, 9, -2, 6]
MATRIX = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
PEOPLE = [
    {"name": "Ada", "dept": "eng", "score": 95, "active": True},
    {"name": "Bo", "dept": "ops", "score": 82, "active": False},
    {"name": "Cy", "dept": "eng", "score": 88, "active": True},
    {"name": "Di", "dept": "ops", "score": 91, "active": True},
]


# --- Part A: convert each of these to a comprehension -------------------------

---

## `a01_uppercase`

_a01 uppercase_

In [ ]:
def a01_uppercase(words: list[str]) -> list[str]:
    out = []
    for w in words:
        out.append(w.upper())
    return out

---

## `a02_positives`

_a02 positives_

In [ ]:
def a02_positives(nums: list[int]) -> list[int]:
    out = []
    for n in nums:
        if n > 0:
            out.append(n)
    return out

---

## `a03_clamp`

_a03 clamp_

In [ ]:
def a03_clamp(nums: list[int]) -> list[int]:
    out = []
    for n in nums:
        out.append(n if n > 0 else 0)
    return out

---

## `a04_lengths`

_a04 lengths_

In [ ]:
def a04_lengths(words: list[str]) -> dict[str, int]:
    out = {}
    for w in words:
        out[w] = len(w)
    return out

---

## `a05_initials`

_a05 initials_

In [ ]:
def a05_initials(words: list[str]) -> set[str]:
    out = set()
    for w in words:
        out.add(w[0].lower())
    return out

---

## `a06_flatten`

_a06 flatten_

In [ ]:
def a06_flatten(matrix: list[list[int]]) -> list[int]:
    out = []
    for row in matrix:
        for cell in row:
            out.append(cell)
    return out

---

## `a07_transpose`

_a07 transpose_

In [ ]:
def a07_transpose(matrix: list[list[int]]) -> list[list[int]]:
    out = []
    for j in range(len(matrix[0])):
        row = []
        for i in range(len(matrix)):
            row.append(matrix[i][j])
        out.append(row)
    return out

---

## `a08_names_of_active`

_a08 names of active_

In [ ]:
def a08_names_of_active(people: list[dict[str, Any]]) -> list[str]:
    out = []
    for p in people:
        if p["active"]:
            out.append(p["name"])
    return out

---

## `a09_by_name`

_a09 by name_

In [ ]:
def a09_by_name(people: list[dict[str, Any]]) -> dict[str, dict[str, Any]]:
    out = {}
    for p in people:
        out[p["name"]] = p
    return out

---

## `a10_high_scorers_by_dept`

_a10 high scorers by dept_

In [ ]:
def a10_high_scorers_by_dept(people: list[dict[str, Any]]) -> dict[str, list[str]]:
    out: dict[str, list[str]] = {}
    for p in people:
        if p["score"] >= 88:
            if p["dept"] not in out:
                out[p["dept"]] = []
            out[p["dept"]].append(p["name"])
    return out

---

## `a11_pairs`

_a11 pairs_

In [ ]:
def a11_pairs(xs: list[int]) -> list[tuple[int, int]]:
    out = []
    for i, x in enumerate(xs):
        for y in xs[i + 1:]:
            out.append((x, y))
    return out

---

## `a12_word_positions`

_a12 word positions_

In [ ]:
def a12_word_positions(words: list[str]) -> dict[str, int]:
    out = {}
    for i, w in enumerate(words):
        out[w.lower()] = i
    return out

---

## `a13_filter_and_transform`

_a13 filter and transform_

In [ ]:
def a13_filter_and_transform(nums: list[int]) -> list[int]:
    out = []
    for n in nums:
        if n > 0:
            if n % 2 == 0:
                out.append(n * n)
    return out

---

## `a14_invert`

_a14 invert_

In [ ]:
def a14_invert(mapping: dict[str, int]) -> dict[int, str]:
    out = {}
    for k, v in mapping.items():
        out[v] = k
    return out

---

## `a15_running_total`

_a15 running total_

In [ ]:
def a15_running_total(nums: list[int]) -> list[int]:
    out = []
    total = 0
    for n in nums:
        total += n
        out.append(total)
    return out

---

## `b01_side_effects`

WHY IS THIS WRONG?

In [ ]:
def b01_side_effects(words: list[str]) -> None:
    """WHY IS THIS WRONG?"""
    [print(w) for w in words]  # noqa: C416

---

## `b02_too_much`

WHY IS THIS WRONG?

In [ ]:
def b02_too_much(people: list[dict[str, Any]]) -> list[str]:
    """WHY IS THIS WRONG?"""
    return [
        f"{p['name']} ({p['dept']})"
        for p in people
        if p["active"]
        if p["score"] > 80
        for _ in range(1 if p["dept"] == "eng" else 2)
        if p["name"][0] not in "XYZ"
    ]

---

## `b03_hidden_work`

WHY IS THIS WRONG?  (hint: how many times is the expensive call made?)

In [ ]:
def b03_hidden_work(paths: list[str]) -> list[str]:
    """WHY IS THIS WRONG?  (hint: how many times is the expensive call made?)"""
    return [
        expensive(p).upper()
        for p in paths
        if expensive(p) is not None
        if len(expensive(p)) > 3
    ]

---

## `expensive`

Pretend this hits the disk.

In [ ]:
def expensive(path: str) -> str | None:
    """Pretend this hits the disk."""
    return path if path else None

---

## `test_part_a`

_test part a_

In [ ]:
def test_part_a() -> None:
    assert a01_uppercase(["a"]) == ["A"]
    assert a02_positives(NUMS) == [3, 4, 5, 9, 6]
    assert a03_clamp([-1, 2]) == [0, 2]
    assert a04_lengths(["ab"]) == {"ab": 2}
    assert a05_initials(["Apple", "avocado"]) == {"a"}
    assert a06_flatten(MATRIX) == [1, 2, 3, 4, 5, 6, 7, 8, 9]
    assert a07_transpose(MATRIX) == [[1, 4, 7], [2, 5, 8], [3, 6, 9]]
    assert a08_names_of_active(PEOPLE) == ["Ada", "Cy", "Di"]
    assert set(a09_by_name(PEOPLE)) == {"Ada", "Bo", "Cy", "Di"}
    assert a10_high_scorers_by_dept(PEOPLE) == {"eng": ["Ada", "Cy"], "ops": ["Di"]}
    assert a11_pairs([1, 2, 3]) == [(1, 2), (1, 3), (2, 3)]
    assert a12_word_positions(["A", "b"]) == {"a": 0, "b": 1}
    assert a13_filter_and_transform(NUMS) == [16, 36]
    assert a14_invert({"a": 1, "b": 2}) == {1: "a", 2: "b"}
    assert a15_running_total([1, 2, 3]) == [1, 3, 6]
    print("  PASS  part A behaviour preserved")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    test_part_a()
    print("\nNow answer the WHY questions in Part B, in comments.")

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.